<a href="https://colab.research.google.com/github/Octaxx/DLI-Assignment/blob/main/GavinModel_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# === Step 1: Load DatasetCleaning.ipynb from GitHub ===
import requests, nbformat
from IPython import get_ipython


url = "https://raw.githubusercontent.com/Octaxx/DLI-Assignment/refs/heads/main/DatasetCleaning.ipynb"
response = requests.get(url)
notebook = nbformat.reads(response.text, as_version=4)
ipython = get_ipython()

print("⚙️ Running cells from DatasetCleaning.ipynb...\n")
for i, cell in enumerate(notebook.cells):
    if cell.cell_type == 'code':
        try:
            print(f"▶️ Executing cell {i+1}...")
            ipython.run_cell(cell.source)
        except Exception as e:
            print(f"❌ Error in cell {i+1}: {e}")
print("\n✅ All notebook cells executed.")

⚙️ Running cells from DatasetCleaning.ipynb...

▶️ Executing cell 2...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
▶️ Executing cell 3...
📊 BEFORE BALANCING
--------------------------------------------------
Total rows before balancing: 18634
Class balance before balancing:
Email Type
Safe Email        11322
Phishing Email     7312
Name: count, dtype: int64


,Email Text,Email Type,Label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,re : equistar deal tickets are you still avail...,Safe Email,0
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,software at incredibly low prices ( 86 % lower...,Phishing Email,1
5,global risk management operations sally congra...,Safe Email,0
6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0
7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1
8,"we owe you lots of money dear applicant , afte...",Phishing Email,1
9,re : coastal deal - with exxon participation u...,Safe Email,0


▶️ Executing cell 4...

📊 AFTER BALANCING (Oversampling)
--------------------------------------------------
Total rows after balancing: 22644
Class balance after balancing:
Email Type
Phishing Email    11322
Safe Email        11322
Name: count, dtype: int64
▶️ Executing cell 5...

🧾 SAMPLE OF CLEANED & BALANCED DATAFRAME
Total Rows        : 22644
Phishing Emails   : 11322
Safe Emails       : 11322

🧪 Cleaned & Balanced DataFrame (First 5 Rows):


,Email Text,Email Type,Label
0,INVESTMENT SCHOLARS CLUB- bringing you the lat...,Phishing Email,1
1,semantics : il dominio tempo-aspettuale il dom...,Safe Email,0
2,mature mom and her young horny lover ! . . woo...,Phishing Email,1
3,do you own a car ; starting december 7 th ford...,Phishing Email,1
4,rescue you from highprice medicaments and badp...,Phishing Email,1



🎯 Phishing Emails (First 5):


...


✅ Safe Emails (First 5):


,Email Text,Email Type,Label
1,semantics : il dominio tempo-aspettuale il dom...,Safe Email,0
5,http://www.bbc.co.uk/radio1/alt/nireland/ni_te...,Safe Email,0
6,Hi Damian.SuSe has a Sparc version I previousl...,Safe Email,0
7,"hpl nom for may 25 , 2001 ( see attached file ...",Safe Email,0
8,iatl 14 : final cfp the 14th annual meeting - ...,Safe Email,0


▶️ Executing cell 6...

🧠 Sample Extracted Features (First 5):


,Email Preview,char_count,word_count,exclamation_count,uppercase_ratio,has_link,has_login_word,has_html
0,INVESTMENT SCHOLARS CLUB- bringing you the lat...,3893,586,2,0.034,0,1,0
1,semantics : il dominio tempo-aspettuale il dom...,904,154,0,0.000,0,0,0
2,mature mom and her young horny lover ! . . woo...,664,141,1,0.000,0,0,0
3,do you own a car ; starting december 7 th ford...,792,170,0,0.000,0,0,0
4,rescue you from highprice medicaments and badp...,796,163,0,0.000,0,0,0



✅ Final Columns:
['Email Text', 'Email Type', 'Label', 'Email Preview', 'char_count', 'word_count', 'exclamation_count', 'uppercase_ratio', 'has_link', 'has_login_word', 'has_html']

✅ All notebook cells executed.


In [3]:
111# === Step 2: Set Number of Epochs and Training Runs ===
try:
    n_epochs = int(input("Enter number of epochs: "))
except:
    n_epochs = 5
    print("⚠️ Invalid input. Defaulting epochs to 5.")

try:
    num_runs = int(input("Enter number of ensemble training runs: "))
except:
    num_runs = 3
    print("⚠️ Invalid input. Defaulting ensemble runs to 3.")

Enter number of epochs: 10
Enter number of ensemble training runs: 10


In [6]:
!pip install pyspellchecker

# === Step 3: Feature Engineering for Classical Models ===
from sklearn.metrics import confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time
from scipy.sparse import hstack, csr_matrix
import pandas as pd
import numpy as np
from spellchecker import SpellChecker


# Copy balanced dataset
df_G = df_balanced.copy()

# ======== ORIGINAL FEATURES ======== #

# Feature extraction from text
df_G['url_count'] = df_G['Email Text'].str.count(r'http[s]?://')
df_G['num_dots'] = df_G['Email Text'].str.count(r'\.')
df_G['has_attachment_word'] = df_G['Email Text'].str.contains(r'attachment|invoice|pdf', case=False).astype(int)
df_G['has_suspicious_word'] = df_G['Email Text'].str.contains(r'verify|account|confirm|login', case=False).astype(int)
df_G['avg_word_length'] = df_G['char_count'] / (df_G['word_count'] + 1)
df_G['special_char_count'] = df_G['Email Text'].str.count(r'[#$%^&*()]')
df_G['has_spam_phrase'] = df_G['Email Text'].str.contains(
    r'congratulations|you have won|limited time|click below', case=False
).astype(int)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
X_tfidf_G = tfidf.fit_transform(df_G['Email Text'])

# Statistical features
stat_features = df_G[[
    'char_count', 'word_count', 'exclamation_count', 'uppercase_ratio',
    'has_link', 'has_login_word', 'has_html', 'url_count', 'num_dots',
    'has_attachment_word', 'has_suspicious_word', 'avg_word_length',
    'special_char_count', 'has_spam_phrase'
]]

# Cue features
def extract_cues(text):
    text = text.lower()
    return pd.Series({
        'has_urgent': int(any(w in text for w in ['urgent', 'immediate', 'now'])),
        'has_threat': int(any(w in text for w in ['suspend', 'locked', 'compromised'])),
        'has_generic': int(any(w in text for w in ['dear user', 'dear customer', 'valued customer']))
    })

cue_features = df_G['Email Text'].apply(extract_cues)

# ======== ADDITIONAL FEATURES ======== #

# 1. DOMAIN-BASED FEATURES
df_G['has_ip_in_url'] = df_G['Email Text'].str.contains(r'http[s]?://(?:\d{1,3}\.){3}\d{1,3}', regex=True).astype(int)
df_G['has_login_url'] = df_G['Email Text'].str.contains(r'http[s]?://[^\s]*login', case=False).astype(int)
df_G['has_uncommon_tld'] = df_G['Email Text'].str.contains(r'\.(xyz|top|tk|pw|club)', case=False).astype(int)


def extract_domain_length(text):
    urls = re.findall(r'http[s]?://([^\s/]+)', text)
    return np.mean([len(url) for url in urls]) if urls else 0

df_G['domain_length'] = df_G['Email Text'].apply(extract_domain_length)

# 2. LINGUISTIC FEATURES
spell = SpellChecker()

def count_spelling_errors(text):
    words = re.findall(r'\b[a-zA-Z]{2,}\b', text)
    misspelled = spell.unknown(words)
    return len(misspelled)

df_G['spelling_errors'] = df_G['Email Text'].apply(count_spelling_errors)

def stopword_ratio(text):
    stopwords = set([
        'the', 'is', 'in', 'and', 'to', 'of', 'for', 'on', 'with', 'that', 'this', 'it', 'as', 'at', 'by', 'an', 'be'
    ])
    words = re.findall(r'\b[a-zA-Z]{2,}\b', text.lower())
    if len(words) == 0:
        return 0
    stop_count = sum(1 for word in words if word in stopwords)
    return stop_count / len(words)

df_G['stopword_ratio'] = df_G['Email Text'].apply(stopword_ratio)

def text_entropy(text):
    prob = [float(text.count(c)) / len(text) for c in dict.fromkeys(list(text))]
    entropy = -sum(p * np.log2(p) for p in prob)
    return entropy

df_G['text_entropy'] = df_G['Email Text'].apply(text_entropy)

# 4. STRUCTURAL / HTML FEATURES
df_G['iframe_count'] = df_G['Email Text'].str.count(r'<iframe')
df_G['form_count'] = df_G['Email Text'].str.count(r'<form')
df_G['html_to_text_ratio'] = df_G['Email Text'].apply(lambda x: len(re.findall(r'<.*?>', x)) / (len(x) + 1))

# Combine original + new statistical features
additional_features = df_G[[
    'has_ip_in_url', 'domain_length',
    'spelling_errors', 'stopword_ratio', 'text_entropy',
    'iframe_count', 'form_count', 'html_to_text_ratio'
]]

full_stat_features = pd.concat([stat_features, additional_features], axis=1)

# Scale stat + cue features
stat_scaled = MinMaxScaler().fit_transform(full_stat_features)
cue_scaled = MinMaxScaler().fit_transform(cue_features)

# Combine everything into final matrix
X_combined = hstack([
    X_tfidf_G,
    csr_matrix(stat_scaled),
    csr_matrix(cue_scaled)
])

# Label column
y = df_G['Label']

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 41.6 MB/s eta 0:00:00


/tmp/ipython-input-1983713755.py:67: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_G['has_uncommon_tld'] = df_G['Email Text'].str.contains(r'\.(xyz|top|tk|pw|club)', case=False).astype(int)


In [7]:
# === Step 4: LSTM Training ===
print("\n\U0001f9e0 Starting LSTM Training...")

max_words = 5000
max_len = 100
embedding_dim = 64

tokenizer_G = Tokenizer(num_words=max_words)
tokenizer_G.fit_on_texts(df_G['Email Text'])
sequences_G = tokenizer_G.texts_to_sequences(df_G['Email Text'])
X_seq_G = pad_sequences(sequences_G, maxlen=max_len)
y_seq_G = df_G['Label'].values

X_train_seq_G, X_test_seq_G, y_train_seq_G, y_test_seq_G = train_test_split(X_seq_G, y_seq_G, test_size=0.2, stratify=y_seq_G)

input_layer = Input(shape=(max_len,))
embedding_layer = Embedding(input_dim=max_words, output_dim=embedding_dim)(input_layer)
lstm_layer = LSTM(64, dropout=0.2, recurrent_dropout=0.2)(embedding_layer)
output_layer = Dense(1, activation='sigmoid')(lstm_layer)

model_LSTM = Model(inputs=input_layer, outputs=output_layer)
model_LSTM.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

history_G = model_LSTM.fit(X_train_seq_G, y_train_seq_G, epochs=n_epochs, batch_size=16,
                    validation_data=(X_test_seq_G, y_test_seq_G), verbose=1)

loss, acc_G = model_LSTM.evaluate(X_test_seq_G, y_test_seq_G, verbose=0)
print(f"\n✅ LSTM Accuracy after {n_epochs} epochs: {acc_G:.4f}")

# === Step 5: Hybrid Model Training (LSTM + Engineered Features + Tree Models) ===
print("\n🤖 Running Hybrid LSTM + Engineered Feature + Tree Models Training...\n")
all_conf_matrices = []

# Extract LSTM feature vector from the trained LSTM model
feature_extractor = Model(inputs=model_LSTM.input, outputs=lstm_layer)
X_lstm_features = feature_extractor.predict(X_seq_G)
y_lstm_labels = y_seq_G

# Combine LSTM features with engineered features (stat_scaled and cue_scaled)
X_combined_hybrid = np.hstack([
    X_lstm_features,
    stat_scaled,  # From MinMaxScaler on stat + new features
    cue_scaled    # From MinMaxScaler on cue features
])

results_G = []

for run in range(num_runs):
    print(f"\n📦 Hybrid Training Run {run + 1}/{num_runs}")
    start = time.time()

    # Split combined feature set
    X_train_G, X_test_G, y_train_G, y_test_G = train_test_split(
        X_combined_hybrid, y_lstm_labels, test_size=0.2, random_state=run, stratify=y_lstm_labels
    )

    # Initialize classifiers
    clf_dt = DecisionTreeClassifier(max_depth=10, random_state=42)
    clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)

    # Ensemble: Decision Tree + Random Forest
    ensemble_G = VotingClassifier(estimators=[
        ('dt', clf_dt),
        ('rf', clf_rf)
    ], voting='soft')

    # Train ensemble model
    ensemble_G.fit(X_train_G, y_train_G)
    y_pred_G = ensemble_G.predict(X_test_G)
    y_proba_G = ensemble_G.predict_proba(X_test_G)[:, 1]

    # Evaluation
    acc_G = accuracy_score(y_test_G, y_pred_G)
    prec_G = precision_score(y_test_G, y_pred_G)
    rec_G = recall_score(y_test_G, y_pred_G)
    f1_G = f1_score(y_test_G, y_pred_G)
    roc_G = roc_auc_score(y_test_G, y_proba_G)
    elapsed = time.time() - start

    print(f"✅ Accuracy: {acc_G:.4f}, Precision: {prec_G:.4f}, Recall: {rec_G:.4f}, F1: {f1_G:.4f}, ROC AUC: {roc_G:.4f}, Time: {elapsed:.2f}s")

    results_G.append({
        'Run': run + 1, 'Accuracy': acc_G, 'Precision': prec_G,
        'Recall': rec_G, 'F1': f1_G, 'ROC_AUC': roc_G, 'Time': elapsed
    })

    cm = confusion_matrix(y_test_G, y_pred_G)
    all_conf_matrices.append(cm)

# Convert results to DataFrame and display
summary_df = pd.DataFrame(results_G)
from IPython.display import display
display(summary_df)

print("\n📊 Averages:")
print(summary_df.mean(numeric_only=True).round(4))


🧠 Starting LSTM Training...
Epoch 1/10
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 128s 110ms/step - accuracy: 0.8589 - loss: 0.3168 - val_accuracy: 0.9291 - val_loss: 0.2987
Epoch 2/10
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 123s 109ms/step - accuracy: 0.9653 - loss: 0.1020 - val_accuracy: 0.9759 - val_loss: 0.0683
Epoch 3/10
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 122s 108ms/step - accuracy: 0.9699 - loss: 0.0814 - val_accuracy: 0.9753 - val_loss: 0.0680
Epoch 4/10
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 143s 108ms/step - accuracy: 0.9818 - loss: 0.0492 - val_accuracy: 0.9762 - val_loss: 0.0693
Epoch 5/10
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 142s 108ms/step - accuracy: 0.9852 - loss: 0.0407 - val_accuracy: 0.9773 - val_loss: 0.0733
Epoch 6/10
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 142s 109ms/step - accuracy: 0.9845 - loss: 0.0394 - val_accuracy: 0.9768 - val_loss: 0.0737
Epoch 7/10
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 143s 109ms/step - accuracy: 0.9882 - loss: 0.0308 - val_accuracy: 0.9764 - val_loss: 0.0867
Epoch 8/10
1133/1133 ━━━━━━━━━━━━

,Run,Accuracy,Precision,Recall,F1,ROC_AUC,Time
0,1,0.988298,0.978779,0.998233,0.988410,0.998757,17.265985
1,2,0.987194,0.979991,0.994702,0.987292,0.998341,15.277091
2,3,0.986531,0.977046,0.996466,0.986661,0.998977,16.032696
3,4,0.986531,0.977046,0.996466,0.986661,0.998686,15.842932
4,5,0.985869,0.975779,0.996466,0.986014,0.998749,16.194148
5,6,0.985427,0.976593,0.994702,0.985564,0.998794,15.092320
6,7,0.984986,0.974093,0.996466,0.985153,0.998281,16.005884
7,8,0.984986,0.975747,0.994702,0.985133,0.998571,15.836797
8,9,0.986752,0.976654,0.997351,0.986894,0.999059,15.820670
9,10,0.987194,0.979149,0.995583,0.987297,0.998380,15.991253



📊 Averages:
Run           5.5000
Accuracy      0.9864
Precision     0.9771
Recall        0.9961
F1            0.9865
ROC_AUC       0.9987
Time         15.9360
dtype: float64
